# File Name Predictor — Neural Network (v3)

Predicts `file_name` from mixed text + numeric features using a multi-input neural network.

**Features:**
- Text columns → `TextVectorization` + `Embedding` + `GlobalAveragePooling1D`
- Numeric columns → `StandardScaler` + `Dense`

**Target:** `file_name` (multi-class classification)

In [ ]:
!pip install tensorflow scikit-learn pandas numpy

In [3]:
import os
os.environ['PYTHONUTF8'] = '1'

import json
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

## 1. Load JSON → DataFrame

In [4]:
JSON_PATH = 'output.json'  # <- change to your file path

with open(JSON_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

df = pd.DataFrame(raw) if isinstance(raw, list) else pd.DataFrame(raw)

print(df.shape)
df.head()

(765366, 13)


,category,file_name,verification_finished,vulnerable_line,column,function,violated_property,stack_trace,error_type,code_snippet,source_code,num_lines,cyclomatic_complexity
0,VULNERABLE,gpt4o_mini,yes,60,9,main,\n file SCRIPT4.c line 60 column 9 function m...,\n c:@F@main\n buffer overflow on scanf\n 0,buffer overflow on scanf,\n,DATASET v1.0 Category: Dice Roller ; Style: li...,74,3.0
1,VULNERABLE,gpt4o_mini,yes,53,9,main,\n file SCRIPT4.c line 53 column 9 function m...,\n c:@F@main\n buffer overflow on scanf\n 0,buffer overflow on scanf,\n,DATASET v1.0 Category: Dice Roller ; Style: li...,74,3.0
2,VULNERABLE,gpt4o_mini,yes,42,9,main,\n file SCRIPT4.c line 42 column 9 function m...,\n c:@F@main\n buffer overflow on scanf\n 0,buffer overflow on scanf,{,DATASET v1.0 Category: Dice Roller ; Style: li...,74,3.0
3,VULNERABLE,gemini_pro,yes,27,49,add_password,\n file SCRIPT5.c line 27 column 49 function ...,\n c:@F@add_password at file gemini_pro-14924...,dereference failure: invalid pointer,e,DATASET v1.0 Category: Password management ; S...,99,2.0
4,VULNERABLE,gemini_pro,yes,26,5,add_password,\n file SCRIPT5.c line 26 column 5 function a...,\n c:@F@add_password at file gemini_pro-14924...,dereference failure: NULL pointer,\n,DATASET v1.0 Category: Password management ; S...,99,2.0


## 2. Define columns

In [5]:
TARGET = 'file_name'

TEXT_COLS = [
    'category',
    'verification_finished',
    'vulnerable_line',
    'function',
    'violated_property',
    'stack_trace',
    'error_type',
    'code_snippet',
    'source_code',
]

NUMERIC_COLS = ['cyclomatic_complexity', 'num_lines', 'column']

# Fill nulls without dropping any rows
df[TEXT_COLS]    = df[TEXT_COLS].fillna('').astype(str)
df[NUMERIC_COLS] = df[NUMERIC_COLS].fillna(0).astype(float)

## 3. Encode target label

In [6]:
label_enc   = LabelEncoder()
y           = label_enc.fit_transform(df[TARGET].astype(str)).astype(np.int32)
num_classes = len(label_enc.classes_)

print(f'Unique file names (classes): {num_classes}')

Unique file names (classes): 9


## 4. Train / test split

In [7]:
idx = np.arange(len(df))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42)

df_train = df.iloc[idx_train].reset_index(drop=True)
df_test  = df.iloc[idx_test].reset_index(drop=True)
y_train  = y[idx_train]
y_test   = y[idx_test]

print(f'Train: {len(df_train)}  |  Test: {len(df_test)}')

Train: 612292  |  Test: 153074


## 5. Build TextVectorization layers

`TextVectorization` lives in `tf.keras.layers` — no legacy preprocessing import needed.
Each column gets its own vectorizer so vocabularies stay separate.

In [8]:
MAX_VOCAB = 100_000
EMBED_DIM = 32

SEQ_LEN = {
    'category':              32,
    'verification_finished': 32,
    'vulnerable_line':       32,
    'function':              64,
    'violated_property':     64,
    'stack_trace':          128,
    'error_type':            32,
    'code_snippet':         128,
    'source_code':          256,
}

vectorizers = {}

for col in TEXT_COLS:
    tv = layers.TextVectorization(
        max_tokens=MAX_VOCAB,
        output_mode='int',
        output_sequence_length=SEQ_LEN[col],
        name=f'tv_{col}',
    )
    tv.adapt(df_train[col].values)
    vectorizers[col] = tv
    print(f'{col:25s}  vocab={tv.vocabulary_size():6d}  seqlen={SEQ_LEN[col]}')

category                   vocab=     3  seqlen=32
verification_finished      vocab=     4  seqlen=32
vulnerable_line            vocab=   355  seqlen=32
function                   vocab= 20363  seqlen=64
violated_property          vocab=100000  seqlen=64
stack_trace                vocab=100000  seqlen=128
error_type                 vocab=    81  seqlen=32
code_snippet               vocab=   737  seqlen=128
source_code                vocab=100000  seqlen=256


## 6. Scale numeric features

In [9]:
scaler      = StandardScaler()
X_num_train = scaler.fit_transform(df_train[NUMERIC_COLS]).astype(np.float32)
X_num_test  = scaler.transform(df_test[NUMERIC_COLS]).astype(np.float32)

print('Numeric train shape:', X_num_train.shape)

Numeric train shape: (612292, 3)


## 7. Build multi-input model

Each text column: `Input (string) → TextVectorization → Embedding → GlobalAveragePooling1D`  
Numeric branch: `Input → Dense(16)`  
All branches concatenated → `Dense(256) → Dropout → Dense(128) → Dropout → Softmax`

In [10]:
text_inputs  = []
text_outputs = []

for col in TEXT_COLS:
    tv     = vectorizers[col]
    inp    = keras.Input(shape=(1,), dtype=tf.string, name=f'input_{col}')
    vec    = tv(inp)
    emb    = layers.Embedding(
                input_dim=tv.vocabulary_size(),
                output_dim=EMBED_DIM,
                name=f'embed_{col}'
             )(vec)
    pooled = layers.GlobalAveragePooling1D(name=f'pool_{col}')(emb)
    text_inputs.append(inp)
    text_outputs.append(pooled)

# Numeric branch
num_inp = keras.Input(shape=(len(NUMERIC_COLS),), dtype=tf.float32, name='input_numeric')
num_x   = layers.Dense(16, activation='relu')(num_inp)

# Merge
merged = layers.Concatenate()(text_outputs + [num_x])
x      = layers.Dense(256, activation='relu')(merged)
x      = layers.Dropout(0.3)(x)
x      = layers.Dense(128, activation='relu')(x)
x      = layers.Dropout(0.2)(x)
output = layers.Dense(num_classes, activation='softmax', name='output')(x)

model = keras.Model(inputs=text_inputs + [num_inp], outputs=output)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_category      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_verification… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_vulnerable_l… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_function      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_violated_pro… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_stack_trace   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_error_type    │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_code_snippet  │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_source_code   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tv_category         │ (None, 32)        │          0 │ input_category[0… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tv_verification_fi… │ (None, 32)        │          0 │ input_verificati… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tv_vulnerable_line  │ (None, 32)        │          0 │ input_vulnerable… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tv_function         │ (None, 64)        │          0 │ input_function[0… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tv_violated_proper… │ (None, 64)        │          0 │ input_violated_p… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tv_stack_trace      │ (None, 128)       │          0 │ input_stack_trac… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tv_error_type       │ (None, 32)        │          0 │ input_error_type… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tv_code_snippet     │ (None, 128)       │          0 │ input_code_snipp

 Total params: 10,401,577 (39.68 MB)

 Trainable params: 10,401,577 (39.68 MB)

 Non-trainable params: 0 (0.00 B)

## 8. Prepare model inputs

In [11]:
def build_inputs(df_split, X_numeric):
    text_arrs = [
        tf.constant(df_split[col].values.reshape(-1, 1), dtype=tf.string)
        for col in TEXT_COLS
    ]
    return text_arrs + [X_numeric]

train_inputs = build_inputs(df_train, X_num_train)
test_inputs  = build_inputs(df_test,  X_num_test)

## 9. Train

In [12]:
history = model.fit(
    train_inputs, y_train,
    validation_data=(test_inputs, y_test),
    epochs=5,
    batch_size=256,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
    ],
)

Epoch 1/5
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 1390s 570ms/step - accuracy: 0.8179 - loss: 0.5546 - val_accuracy: 0.9292 - val_loss: 0.2209
Epoch 2/5
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 412s 172ms/step - accuracy: 0.9524 - loss: 0.1546 - val_accuracy: 0.9660 - val_loss: 0.1129
Epoch 3/5
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 812s 339ms/step - accuracy: 0.9738 - loss: 0.0848 - val_accuracy: 0.9749 - val_loss: 0.0846
Epoch 4/5
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 147s 61ms/step - accuracy: 0.9822 - loss: 0.0572 - val_accuracy: 0.9694 - val_loss: 0.1025
Epoch 5/5
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 160s 67ms/step - accuracy: 0.9859 - loss: 0.0448 - val_accuracy: 0.9802 - val_loss: 0.0712


## 10. Evaluate

In [13]:
loss, acc = model.evaluate(test_inputs, y_test, verbose=0)
print(f'Test accuracy: {acc:.4f}  |  Test loss: {loss:.4f}')

Test accuracy: 0.9802  |  Test loss: 0.0712


## 11. Save model + encoders

In [ ]:
# Does not work
# model.save('file_name_predictor.keras')

# with open('label_enc.pkl', 'wb') as f:
#     pickle.dump(label_enc, f)

# with open('scaler.pkl', 'wb') as f:
#     pickle.dump(scaler, f)

# print('Model and encoders saved.')

UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f632' in position 2111: character maps to <undefined>

## 12. Load model + encoders (run this in any new file/session instead of retraining)

> Make sure `os.environ['PYTHONUTF8'] = '1'` is set before importing tensorflow in the new file.

In [ ]:
# Does not work
# model     = keras.models.load_model('file_name_predictor.keras')
# label_enc = pickle.load(open('label_enc.pkl', 'rb'))
# scaler    = pickle.load(open('scaler.pkl', 'rb'))

# print('Model and encoders loaded.')

ValueError: Expected a model.weights.h5 or model.weights.npz file.

## 13. Predict on a single row

In [20]:
def predict_file_name(row: dict) -> str:
    """Pass a dict with the same keys as the dataframe columns."""
    text_arrs = [
        tf.constant([[str(row.get(col, ''))]], dtype=tf.string)
        for col in TEXT_COLS
    ]
    num_feat = scaler.transform(pd.DataFrame([[row.get(c, 0) for c in NUMERIC_COLS]], columns=NUMERIC_COLS))
    inputs   = text_arrs + [num_feat.astype(np.float32)]
    probs    = model.predict(inputs, verbose=0)
    return label_enc.inverse_transform([np.argmax(probs)])[0]

# Example: predict on the first test row
for i in range(0, 100):
    sample = df_test.iloc[i].to_dict()
    if(predict_file_name(sample) != sample[TARGET]):
        print()
        print('-------------------------------------------------------------------')
        print('Predicted:', predict_file_name(sample))
        print('Actual:   ', sample[TARGET])
        print('-------------------------------------------------------------------')
        print()
    else:
        print('Predicted:', predict_file_name(sample))
        print('Actual:   ', sample[TARGET])
        print()
        

Predicted: gemini_pro
Actual:    gemini_pro

Predicted: gpt35
Actual:    gpt35

Predicted: falcon180b
Actual:    falcon180b

Predicted: falcon180b
Actual:    falcon180b

Predicted: gemini_pro
Actual:    gemini_pro

Predicted: llama2
Actual:    llama2

Predicted: falcon180b
Actual:    falcon180b

Predicted: falcon180b
Actual:    falcon180b

Predicted: gemma7b
Actual:    gemma7b

Predicted: gpt4o_mini
Actual:    gpt4o_mini

Predicted: llama2
Actual:    llama2

Predicted: gemma7b
Actual:    gemma7b

Predicted: falcon2
Actual:    falcon2

Predicted: gemma7b
Actual:    gemma7b

Predicted: gpt35
Actual:    gpt35

Predicted: falcon180b
Actual:    falcon180b


-------------------------------------------------------------------
Predicted: llama2
Actual:    gpt35
-------------------------------------------------------------------

Predicted: falcon180b
Actual:    falcon180b

Predicted: falcon180b
Actual:    falcon180b

Predicted: gemini_pro
Actual:    gemini_pro

Predicted: gemini_pro
Actual:   